# Regression — Guided Examples

This notebook walks through a practical regression workflow using synthetic data. Run cells from top to bottom. The dataset is generated locally and is for learning only.

**Workflow:** generate data → baseline → fit model → evaluate → inspect residuals → compare regularization → cross-validate.

## 1. Imports and reproducibility

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
print("Random state:", RANDOM_STATE)

## 2. Generate a synthetic regression dataset

In [ ]:
X_array, y = make_regression(
    n_samples=1200, n_features=6, n_informative=4,
    noise=22.0, random_state=RANDOM_STATE
)
feature_names = [f"feature_{i}" for i in range(X_array.shape[1])]
df = pd.DataFrame(X_array, columns=feature_names)
df["target"] = y
df.head()

## 3. Inspect the dataset

In [ ]:
print("Shape:", df.shape)
display(df.describe().T)
print("Missing values:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))

In [ ]:
df["target"].hist(bins=35)
plt.xlabel("Target")
plt.ylabel("Count")
plt.title("Target distribution")
plt.show()

## 4. Split data and define a baseline
The test set is held out for final evaluation. For time-dependent data, use a chronological split rather than a random split.

In [ ]:
X = df.drop(columns="target")
y = df["target"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

## 5. Fit linear regression

In [ ]:
linear = LinearRegression()
linear.fit(X_train, y_train)
pred = linear.predict(X_test)

def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "MSE": mean_squared_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    }

comparison = pd.DataFrame({
    "Mean baseline": regression_metrics(y_test, baseline_pred),
    "Linear regression": regression_metrics(y_test, pred),
}).T
comparison

## 6. Interpret coefficients cautiously

In [ ]:
coefficients = pd.Series(linear.coef_, index=X.columns, name="coefficient")
display(coefficients.sort_values(key=np.abs, ascending=False).to_frame())
print("Intercept:", linear.intercept_)

Coefficients describe fitted relationships in this synthetic dataset. In observational data, a coefficient alone does not prove that changing a feature causes the target to change.

## 7. Observed vs. predicted

In [ ]:
plt.scatter(y_test, pred, alpha=0.6)
low = min(y_test.min(), pred.min())
high = max(y_test.max(), pred.max())
plt.plot([low, high], [low, high], linestyle="--")
plt.xlabel("Observed target")
plt.ylabel("Predicted target")
plt.title("Observed vs. predicted — test set")
plt.show()

## 8. Residual analysis

In [ ]:
residuals = y_test - pred
plt.scatter(pred, residuals, alpha=0.6)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted target")
plt.ylabel("Residual (observed - predicted)")
plt.title("Residuals vs. predictions")
plt.show()

print("Residual mean:", round(float(residuals.mean()), 3))
print("Residual standard deviation:", round(float(residuals.std()), 3))

Look for curvature, funnel-shaped variance, clusters, and unusually large errors. Residual plots complement summary metrics.

## 9. Ridge and Lasso regularization

In [ ]:
ridge = make_pipeline(StandardScaler(), Ridge(alpha=10.0))
lasso = make_pipeline(StandardScaler(), Lasso(alpha=1.0, max_iter=10000))
ridge.fit(X_train, y_train)
lasso.fit(X_train, y_train)
ridge_pred = ridge.predict(X_test)
lasso_pred = lasso.predict(X_test)

regularized_results = pd.DataFrame({
    "Linear": regression_metrics(y_test, pred),
    "Ridge": regression_metrics(y_test, ridge_pred),
    "Lasso": regression_metrics(y_test, lasso_pred),
}).T
regularized_results

The `alpha` values above are illustrative, not guaranteed to be optimal. Tune hyperparameters with cross-validation on training data.

## 10. Polynomial regression example
This example uses one feature so the relationship can be plotted. High polynomial degrees can overfit.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
x = np.linspace(-3, 3, 250)
y_curve = 2 + 1.5*x - 0.8*x**2 + rng.normal(0, 1.3, size=len(x))
X_curve = x.reshape(-1, 1)
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_curve, y_curve, test_size=0.2, random_state=RANDOM_STATE
)

degree_results = {}
degree_models = {}
for degree in [1, 2, 5]:
    estimator = make_pipeline(
        PolynomialFeatures(degree=degree, include_bias=False),
        LinearRegression()
    )
    estimator.fit(Xc_train, yc_train)
    degree_models[degree] = estimator
    degree_results[degree] = {
        "train_RMSE": np.sqrt(mean_squared_error(yc_train, estimator.predict(Xc_train))),
        "test_RMSE": np.sqrt(mean_squared_error(yc_test, estimator.predict(Xc_test))),
    }

pd.DataFrame(degree_results).T

In [ ]:
x_grid = np.linspace(-3, 3, 300).reshape(-1, 1)
plt.scatter(Xc_train.ravel(), yc_train, s=15, alpha=0.45, label="Train")
for degree, estimator in degree_models.items():
    plt.plot(x_grid.ravel(), estimator.predict(x_grid), label=f"Degree {degree}")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Polynomial model comparison")
plt.legend()
plt.show()

## 11. Cross-validation on the training data

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
candidates = {
    "Linear": make_pipeline(StandardScaler(), LinearRegression()),
    "Ridge": make_pipeline(StandardScaler(), Ridge(alpha=10.0)),
    "Lasso": make_pipeline(StandardScaler(), Lasso(alpha=1.0, max_iter=10000)),
}
cv_rows = {}
for name, estimator in candidates.items():
    scores = cross_validate(
        estimator, X_train, y_train, cv=cv,
        scoring=["neg_mean_absolute_error", "neg_root_mean_squared_error", "r2"]
    )
    cv_rows[name] = {
        "MAE": -scores["test_neg_mean_absolute_error"].mean(),
        "RMSE": -scores["test_neg_root_mean_squared_error"].mean(),
        "R2": scores["test_r2"].mean(),
    }
pd.DataFrame(cv_rows).T

## 12. Error analysis

In [ ]:
error_table = X_test.copy()
error_table["actual"] = y_test
error_table["predicted"] = pred
error_table["residual"] = error_table["actual"] - error_table["predicted"]
error_table["absolute_error"] = error_table["residual"].abs()
error_table.sort_values("absolute_error", ascending=False).head(10)

Inspect large errors for data quality problems or systematic gaps. Do not automatically delete observations just because the model performs poorly on them.

## 13. Experiment record

Record the target and units, data source, split strategy, baseline, model, metrics, diagnostics, and limitations. For a real deployment, also define monitoring and retraining triggers.

### Extension tasks
- Try a different `Ridge(alpha=...)` and `Lasso(alpha=...)` value using cross-validation.
- Add missing values and build a pipeline with `SimpleImputer`.
- Add categorical features and use `ColumnTransformer` plus `OneHotEncoder`.
- Use a time-based split for chronological data.
- Explain which metric best reflects the real-world cost of prediction errors.

**Reminder:** synthetic results demonstrate methods, not real-world model performance.